# Task5 — Context-matched Behavioral Blocks

พัฒนาต่อจาก density baseline โดยเปลี่ยนเฉพาะ HGB component: ข้อมูลครบ 4 ช่วง, สลับกลุ่มพฤติกรรมภายในบริบท, และให้ความไม่สอดคล้องเป็นคะแนน anomaly สูง

คง Within-account Change และ Univariate Density Modes และ budget 2,046 alerts. ไม่มี anomaly labels; ค่าเริ่มต้น Run All โหลดผล ไม่ฝึกหรือส่ง Kaggle

**ผลตรวจ: ยังไม่แนะนำส่งรอบสุดท้าย** — auxiliary AUC 0.500473 และ HGB-only bag Jaccard 1.23%; หลักฐานยังไม่สนับสนุนความเสถียรของ component ใหม่

## 1. Setup

In [1]:
from pathlib import Path
import sys, json, os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
ROOT = next((p for q in [Path.cwd(), *Path.cwd().parents] for p in [q, q / "CPE342-Kaggle"] if (p / "Task5/models/behavior_blocks_followup.py").exists()), None)
if ROOT is None: raise FileNotFoundError("เปิด notebook ภายใน project CPE342-Kaggle")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from Task5.models import behavior_blocks_followup as experiment
from Task5.models import behavior_blocks_report as report
RUN_TRAINING = False
RUN_FULL_VERIFICATION = False
EXPORT_CSV = False
print(ROOT)

/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle


## 2. Data & Feature Blocks

In [2]:
raw, x, source, stage, baseline, context = experiment.inputs()
display(pd.Series({"accounts":len(raw), "raw_features":x.shape[1], "HGB_features":len(experiment.FEATURE_COLUMNS), "background_accounts":int(stage["background"].sum())}))
display(pd.DataFrame([{"block":g, "columns":len(idx), "families":", ".join(experiment.BLOCK_FAMILIES[g])} for g,idx in experiment.BLOCKS.items()]))

accounts               25889
raw_features             122
HGB_features             202
background_accounts    24098
dtype: int64

,block,columns,families
0,access,35,"location_changes, device_count, ip_hash_entrop..."
1,engagement,35,"login_count, session_duration_mean, session_du..."
2,performance,64,"skill_rating, skill_trajectory, kda_mean, kda_..."
3,economy,35,"purchase_frequency, purchase_amount_mean, purc..."
4,social,28,"friend_adds, friend_removes, communication_fre..."
5,security_flags,4,"password_changed, vpn_usage, suspicious_login_..."


202 row-local features: asinh-transformed valuesครบทั้ง4ช่วง + Haar contrasts + ระยะทางครบ6คู่ + invalid-coordinate count.
บัญชีที่มี invalid geo ไม่ถูก label ว่า anomaly อัตโนมัติ. Missing คงไว้ใน raw HGB features, median/scaler ใช้เฉพาะ sparse projectionsและfitเฉพาะ training background.

สลับทุกคอลัมน์ใน block ด้วย donor เดียวกัน จึงรักษาลำดับเวลา, Haar identities และ missing pattern ภายใน block. account_age_days คงค่าบัญชีเดิม.
Static age + mean activity periods1–3 กำหนด quartile strata; cutpoints fitเฉพาะtraining background. เป็นบริบทใกล้เคียง ไม่ใช่อายุ/activityเท่ากันเป๊ะ

## 3. Hypothesis & Controlled Experiment

**Candidate block5:** แยกข้อมูลจริงออกจากข้อมูลที่ถูกสลับความสัมพันธ์ข้ามกลุ่มพฤติกรรม.

**Control column_control:** ฟีเจอร์/บริบท/folds/seeds/HGB/calibration เหมือนกัน แต่สลับรายคอลัมน์ ซึ่งอาจทำลายสมการระหว่างค่ารายช่วงกับ derived contrasts. AUC สูงของ control จึงอาจแปลว่าจับข้อมูลจำลองที่เป็นไปไม่ได้ได้ง่าย.

ทั้งสองตัวเปลี่ยนเฉพาะ HGB component ของ pipeline รวม; เทียบตัวดีที่สุดแล้ว HGB เปลี่ยนทั้ง representation, null generation, score direction และ calibration reference. การเทียบ control กับ candidate จึงใช้แยกผล block corruptionโดยเฉพาะ

In [3]:
display(pd.Series(experiment.CONFIG))

folds                                                                      5
bags                                                                       5
projections                                                              140
iterations                                                               160
training_null_repeats                                                      3
evaluation_null_repeats                                                    2
seed                                                                     342
threads                                                                    4
max_bins                                                                  63
learning_rate                                                           0.07
max_depth                                                                  5
l2_regularization                                                        2.0
context_features                   [account_age_days, activity_history_mean]

## 4. Train / Resume — 5 Folds × 5 Bags × 2 Arms

In [4]:
experiment.initialize()
if RUN_TRAINING:
    experiment.train("column_control")
    experiment.train("block5")
    experiment.report()
else:
    assert (experiment.OUT / "summary.json").exists(), "เปิด RUN_TRAINING หากยังไม่มีผลฝึก"
logs=[]
for mode in ["column_control", "block5"]:
    for item in json.loads((experiment.OUT / f"{mode}_training_log.json").read_text()): logs.append(item["record"])
display(pd.DataFrame(logs))

,mode,bag,fold,seconds,auxiliary_auc,auc_note,training_strata,min_training_stratum_rows,singleton_training_rows
0,column_control,1,1,3.208032,0.999330,"Real-vs-null discrimination, NOT fraud AUC",21,1,1
1,column_control,1,2,3.138646,0.999385,"Real-vs-null discrimination, NOT fraud AUC",20,229,0
2,column_control,1,3,3.214227,0.999223,"Real-vs-null discrimination, NOT fraud AUC",21,1,1
3,column_control,1,4,3.148394,0.999293,"Real-vs-null discrimination, NOT fraud AUC",21,1,1
4,column_control,1,5,3.141666,0.998762,"Real-vs-null discrimination, NOT fraud AUC",21,1,1
5,column_control,2,1,3.233024,0.999888,"Real-vs-null discrimination, NOT fraud AUC",21,1,1
6,column_control,2,2,3.335059,0.999694,"Real-vs-null discrimination, NOT fraud AUC",20,229,0
7,column_control,2,3,3.370436,0.999838,"Real-vs-null discrimination, NOT fraud AUC",21,1,1
8,column_control,2,4,3.456782,0.999930,"Real-vs-null discrimination, NOT fraud AUC",21,1,1
9,column_control,2,5,3.483689,0.999681,"Real-vs-null discrimination, NOT fraud AUC",21,1,1


มี50 HGB fits, checkpoint เมื่อจบแต่ละfold. รันใหม่โหลดfoldที่เสร็จแล้ว; foldที่หยุดกลางทางต้องเริ่มใหม่.
Real=1/corrupted=0 เป็น label ของงานช่วยฝึก ไม่ใช่ normal/anomaly labels.
Shared baseline geometry/background เป็น transductive; ตัวclassifier/preprocessing/context cutsไม่fitบนheld-out account

## 5. Score Direction & Calibration

HGB real-vs-corrupted log odds ต่ำ → คล้ายข้อมูลที่ไม่สอดคล้อง → ตั้ง inconsistency = **negative log odds**.

ใช้ ExcessMassCalibration เปรียบเทียบ inconsistency ของทุกบัญชีกับ **OOF real background** เพื่อประมาณ excess mass. ไม่ใช้ corrupted rows เป็น normal reference.
Background อาจปน anomaly และถูกเลือกด้วยโมเดลเก่า จึงเป็น heuristic ไม่ใช่ calibrated fraud probability.

คง shift และ island probabilities เดิมทุกแถว, รวมด้วยสูตรเดิม, ใช้อันดับ top2,046. ไม่มีการ optimize F3 จริงในเครื่อง

In [5]:
summary=json.loads((experiment.OUT / "summary.json").read_text())
display(pd.DataFrame(summary["comparisons"]))
display(pd.Series({k:v for k,v in summary.items() if k!="comparisons"}))

,mode,alerts,retained,added,removed,original_supported_alerts_removed,jaccard,measured_f3
0,column_control,2046,1795,251,251,0,0.781454,None
1,block5,2046,1798,248,248,0,0.783784,None


paired_block_vs_column_added                                                    254
paired_block_vs_column_removed                                                  254
model_count                                                                      50
warning                           No anomaly labels. Real-vs-corrupted AUC and a...
public_score                                                                   None
candidate_predeclared                                                        block5
dtype: object

## 6. Diagnostics — Not Fraud Accuracy

In [6]:
diagnostics=json.loads((experiment.OUT / "diagnostics.json").read_text())
display(pd.DataFrame(diagnostics["models"]))
display(pd.DataFrame(diagnostics["reference_comparisons"]))

,mode,mean_auxiliary_auc,mean_bag_alert_jaccard,mean_bag_hgb_only_alert_jaccard,total_fit_seconds,interpretation
0,column_control,0.999608,0.824467,0.136045,85.716519,AUC measures real-vs-corruption discrimination...
1,block5,0.500473,0.783164,0.012327,87.428894,AUC measures real-vs-corruption discrimination...


,reference,agreement,common_alerts,jaccard,added,removed
0,density_auto_v1,0.980841,1798,0.783784,248,248
1,rank_fusion95_5_v1,0.978524,1768,0.760757,278,278
2,tail_engineered_v20,0.797018,1661,0.240168,385,4870
3,multiscale_representation_v1,0.932635,1174,0.402330,872,872


Auxiliary AUC วัดการแยกreal/corrupted; Jaccardวัดความเหมือนระหว่างbags/reference.
ทั้งสองค่าไม่ยืนยัน fraud accuracy หรือคะแนนKaggle. ใช้เพื่อดูพฤติกรรมของวิธี ไม่ใช้ประกาศว่าชนะ0.81771

## 7. Inspect Changed Accounts

In [7]:
audit=pd.read_parquet(experiment.OUT / "row_audit.parquet")
display(audit.groupby("change").agg(accounts=("id","size"),mean_missing=("missing_raw_count","mean")))
display(audit[audit.change!="unchanged"][["id","player_id","change","inconsistency_score","dependence_probability","baseline_shift","baseline_island"]].head(30))

,accounts,mean_missing
change,,
added,248,5.592742
removed,248,5.967742
unchanged,25393,5.956169


,id,player_id,change,inconsistency_score,dependence_probability,baseline_shift,baseline_island
127,ANS00128,P003002,added,0.264744,0.276805,0.0,0.0
135,ANS00136,P023396,added,0.345106,0.632353,0.0,0.0
206,ANS00207,P010977,removed,-0.135758,0.000000,0.0,0.0
257,ANS00258,P009425,added,0.297500,0.451754,0.0,0.0
312,ANS00313,P025193,added,0.271151,0.276805,0.0,0.0
340,ANS00341,P010520,added,0.361475,0.635417,0.0,0.0
345,ANS00346,P022952,removed,0.048989,0.000000,0.0,0.0
415,ANS00416,P009874,removed,-0.051700,0.000000,0.0,0.0
449,ANS00450,P025736,added,0.280809,0.315476,0.0,0.0
462,ANS00463,P018785,added,0.293274,0.362562,0.0,0.0


## 8. Replay Verification

In [8]:
if RUN_FULL_VERIFICATION: experiment.verify()
display(pd.Series(json.loads((experiment.OUT / "verification.json").read_text())))

bundles                                     50
real_logits_recomputed                  258890
null_logits_recomputed                  481960
block_integrity_and_context_verified      True
shift_and_island_unchanged                True
heldout_disjoint                          True
anomaly_accuracy_verified                False
dtype: object

ตรวจ held-out disjointness, training-only context cuts/imputation, block donor integrity, marginal distributionsรวมNaNภายในstratum, logitsหลังreload, aggregation และคะแนน Within-account Change กับ Univariate Density Modes เดิมไม่เปลี่ยน

## 9. Combined Submission

In [9]:
if EXPORT_CSV: report.export()
display(pd.Series(report.verify_csv()))
print(report.DEST)
print("ยังไม่ส่ง Kaggle; CSV พร้อมใช้งานไม่ได้หมายถึงคะแนนดีขึ้นแน่นอน")

rows                                                                                       25889
alerts                                                                                      2046
ids_and_tasks1_to4_literal_fields_preserved                                                 True
decisions_rebuilt_from_fold_logits                                                          True
source_sha256                                  cd5725c7e9994d60a05ad31f917107d92e268f9833c0c6...
output_sha256                                  e2e99a605837dd50eb5583c1824a8b347758118e8f2eff...
dtype: object

/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/submission/kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_behavior_blocks5_v1.csv
ยังไม่ส่ง Kaggle; CSV พร้อมใช้งานไม่ได้หมายถึงคะแนนดีขึ้นแน่นอน


## 10. Interpretation & Next Decision

ดูจำนวน added/removed และ original_supported_alerts_removed ก่อนใช้โควตาสุดท้าย.
โมเดลอาจมองบัญชีที่เปลี่ยนพฤติกรรมอย่างถูกต้องว่าไม่สอดคล้อง หรือไม่พบผู้โจมตีที่รักษาพฤติกรรมให้สอดคล้อง.
Security flagsเป็นfeatures ไม่ใช่ground truth. Contextมี singleton strata บางfoldซึ่งไม่สามารถสลับdonorได้.

วิธีนี้เป็นการพัฒนาต่อจากbaselineเดิมและใช้ส่วนประกอบมาตรฐาน ไม่ใช่pipelineใหม่ทั้งหมด.
คงไฟล์0.81771ไว้เป็นตัวหลักจนกว่าจะมีหลักฐานคะแนนจริง. ไม่มีการไล่ลองweights/budgetsผ่านKaggleอัตโนมัติ